In [ ]:
import pandas as pd

df = pd.read_csv('../data/raw/arabica_coffee_full_table.csv')

contagem_paises = df['Country_of_Origin'].value_counts()
paises_raros = contagem_paises[contagem_paises < 10].index

df['Country_of_Origin'] = df['Country_of_Origin'].apply(
    lambda x: 'Outros' if x in paises_raros else x
)

# Features que vamos usar (evitando vazamento de dados do Total_Cup_Points)
colunas_numericas = ['Moisture', 'Category_One_Defects', 'Category_Two_Defects', 
                      'Quakers', 'Altitude']
colunas_categoricas = ['Variety', 'Processing_Method', 'Country_of_Origin', 'Color']
coluna_target = 'Total_Cup_Points'

df_modelo = df[colunas_numericas + colunas_categoricas + [coluna_target]].copy()

for col in colunas_numericas:
    df_modelo[col] = df_modelo[col].fillna(df_modelo[col].median())

for col in colunas_categoricas:
    df_modelo[col] = df_modelo[col].fillna('Unknown')

print(df_modelo.isnull().sum())  # confirma que zerou tudo



Quantidade de registros: 1509
Quantidade de colunas: 42


,coffee_id,Country_of_Origin,Farm_Name,Lot_Number,Mill,ICO_Number,Company,Altitude,Region,Producer,...,Color,Category_One_Defects,Category_Two_Defects,Quakers,Expiration,Certification_Body,Certification_Address,Certification_Contact,parsed_expiration,parsed_grading_date
0,#647123,Guatemala,san francisco cotzal,11/441/50,"inmobiliaria e inversiones dos mil, s.a.",11/441/50,"inmobiliaria e inversiones dos mil, s.a.",1600.0,quiche,san francisco cotzal,...,Green,0,1,3.0,June 22 2023,Asociacion Nacional Del Café,"5a Calle 0-50, Zona 14 Guatemala City, Guatema...",Brayan Cifuentes -,2023-06-22,2022-06-22
1,#927000,Guatemala,San jose del lago,11/15/95,San jose del lago,11/15/95,"Peter Schoenfeld, S.A.",1600.0,Atitlán,"Cafetalera Paquim, S.A.",...,Green,0,2,1.0,April 16 2024,Asociacion Nacional Del Café,"5a Calle 0-50, Zona 14 Guatemala City, Guatema...",Brayan Cifuentes -,2024-04-16,2023-04-17
2,#902618,Guatemala,varias fincas,11/15/51,El Trèbol/Lìnea Gourmet,11/15/51,"Peter Schoenfeld, S.A.",1550.0,Oriente Santa rosa,varios productores,...,Green,0,2,1.0,March 21 2024,Asociacion Nacional Del Café,"5a Calle 0-50, Zona 14 Guatemala City, Guatema...",Brayan Cifuentes -,2024-03-21,2023-03-22
3,#781706,Guatemala,San jose del lago,11/15/96,San jose del lago,11/15/96,"Peter Schoenfeld, S.A.",1600.0,Atitlán,"Cafetalera Paquim, S.A.",...,Green,0,1,0.0,April 16 2024,Asociacion Nacional Del Café,"5a Calle 0-50, Zona 14 Guatemala City, Guatema...",Brayan Cifuentes -,2024-04-16,2023-04-17
4,#237025,Guatemala,Finca Alta Luz,11-63-657,NaN,11-63-657,"Retrillas del pacifico, s.a.",1350.0,Huehuetenango,Maria de los Angeles Perez,...,Green,0,5,1.0,April 25 2024,Asociacion Nacional Del Café,"5a Calle 0-50, Zona 14 Guatemala City, Guatema...",Brayan Cifuentes -,2024-04-25,2023-04-26


In [ ]:
print(df_modelo.duplicated().sum())

df_modelo[colunas_numericas].describe()



2


,Moisture,Category_One_Defects,Category_Two_Defects,Quakers,Altitude
count,1509.000000,1509.000000,1509.000000,1509.000000,1509.000000
mean,0.091294,0.388999,3.396952,0.233930,1301.646315
std,0.045368,1.724503,5.115200,0.962983,390.049638
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.098000,0.000000,0.000000,0.000000,1200.000000
50%,0.110000,0.000000,2.000000,0.000000,1311.000000
75%,0.120000,0.000000,4.000000,0.000000,1550.000000
max,0.280000,31.000000,55.000000,11.000000,2560.000000


In [ ]:
df_modelo[df_modelo["Altitude"] == 0]

(df_modelo["Altitude"] == 0).sum()

np.int64(1)

In [ ]:
# Colunas de identificação (só referência, NÃO entram no X)
colunas_id = ['coffee_id', 'Farm_Name', 'Country_of_Origin', 'Region']

# Colunas sensoriais (features de avaliação)
colunas_sensoriais = ['Aroma', 'Flavor', 'Aftertaste', 'Acidity', 
                       'Body', 'Balance', 'Uniformity', 'Clean_Cup', 'Sweetness']

coluna_target = 'Total_Cup_Points'

df_modelo = df[colunas_id + colunas_sensoriais + [coluna_target]].copy()

df_modelo = df_modelo[df_modelo['Total_Cup_Points'] > 0]
print(df_modelo.shape)
df_modelo.head()

(1508, 14)


,coffee_id,Farm_Name,Country_of_Origin,Region,Aroma,Flavor,Aftertaste,Acidity,Body,Balance,Uniformity,Clean_Cup,Sweetness,Total_Cup_Points
0,#647123,san francisco cotzal,Guatemala,quiche,7.83,7.92,7.75,8.00,7.75,7.75,10.0,10.0,10.0,84.92
1,#927000,San jose del lago,Guatemala,Atitlán,7.58,7.83,7.58,7.75,7.67,7.75,10.0,10.0,10.0,83.92
2,#902618,varias fincas,Guatemala,Oriente Santa rosa,7.67,7.83,7.67,7.83,7.75,7.67,10.0,10.0,10.0,84.08
3,#781706,San jose del lago,Guatemala,Atitlán,7.58,7.92,7.67,7.75,7.83,7.75,10.0,10.0,10.0,84.25
4,#237025,Finca Alta Luz,Guatemala,Huehuetenango,7.67,7.83,7.75,7.75,7.83,7.75,10.0,10.0,10.0,84.33


In [ ]:
def classificar(pontos):
    if pontos < 82.4:
        return "Tradicional"
    elif pontos < 84:
        return "Superior"
    else:
        return "Gourmet"

df_modelo['classe'] = df_modelo[coluna_target].apply(classificar)
print(df_modelo['classe'].value_counts())

classe
Tradicional    646
Superior       512
Gourmet        350
Name: count, dtype: int64


In [ ]:
print(df_modelo.groupby('classe')['Total_Cup_Points'].agg(['min', 'max', 'mean', 'count']))
# Ver quantas linhas têm esse problema
print(df_modelo[df_modelo['Total_Cup_Points'] < 50])

               min    max       mean  count
classe                                     
Gourmet      84.00  90.58  85.166000    350
Superior     82.42  83.92  83.132031    512
Tradicional   0.00  82.33  80.169583    647
     coffee_id     Farm_Name Country_of_Origin     Region  Aroma  Flavor  \
1031      1312  los hicaques          Honduras  comayagua    0.0     0.0   

      Aftertaste  Acidity  Body  Balance  Uniformity  Clean_Cup  Sweetness  \
1031         0.0      0.0   0.0      0.0         0.0        0.0        0.0   

      Total_Cup_Points       classe  
1031               0.0  Tradicional  


In [ ]:
df_modelo["Total_Cup_Points"].describe()

count    1509.000000
mean       82.333612
std         3.378540
min         0.000000
25%        81.420000
50%        82.670000
75%        83.830000
max        90.580000
Name: Total_Cup_Points, dtype: float64

In [ ]:
# Salva a tabela processada em uma nova pasta
df_modelo.to_csv('../data/processed/coffee_classificado.csv', index=False)
print("Tabela salva com sucesso!")

Tabela salva com sucesso!
